# ATC-ASR Benchmark

In [ ]:
%pip install -q transformers>=4.35 datasets jiwer soundfile accelerate numpy

In [ ]:
import torch, json, time, re
from pathlib import Path
from collections import Counter
import numpy as np
from datasets import load_dataset
from transformers import pipeline as hf_pipeline
from jiwer import wer, cer, process_words

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path("/content/drive/MyDrive/atc_asr_output")
except Exception:
    OUTPUT_DIR = Path("atc_asr_output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "configs").mkdir(exist_ok=True)

## Normalizer

In [ ]:
DIGIT_TO_WORD = {
    "0":"zero","1":"one","2":"two","3":"three","4":"four",
    "5":"five","6":"six","7":"seven","8":"eight","9":"nine",
}
WORD_TO_DIGIT = {v:k for k,v in DIGIT_TO_WORD.items()}
WORD_TO_DIGIT["niner"] = "9"


def normalize_for_wer(text: str) -> str:
    t = text.lower().strip()
    t = re.sub(r"[,\.!?;:\"\(\)]", " ", t)
    t = re.sub(r"\bniner\b", "nine", t)
    def expand_fl(m):
        return "flight level " + " ".join(DIGIT_TO_WORD[d] for d in m.group(1))
    t = re.sub(r"\bfl\s*(\d{2,3})\b", expand_fl, t)
    def expand_rwy(m):
        side = {"l":"left","r":"right","c":"center"}.get((m.group(2) or "").lower(), "")
        words = " ".join(DIGIT_TO_WORD[d] for d in m.group(1))
        return ("runway " + words + (" " + side if side else "")).strip()
    t = re.sub(r"\brwy\s*(\d{1,2})([lrc]?)\b", expand_rwy, t)
    t = re.sub(r"\b(\d+)\b", lambda m: " ".join(DIGIT_TO_WORD[d] for d in m.group(1)), t)
    return re.sub(r"\s+", " ", t).strip()


def normalize_for_display(text: str) -> str:
    t = text.strip()
    num_words = "|".join(WORD_TO_DIGIT.keys())
    def compress_fl(m):
        ds = re.findall(rf"\b({num_words})\b", m.group(1))
        return "FL" + "".join(WORD_TO_DIGIT[d] for d in ds) if ds else m.group(0)
    t = re.sub(r"\bflight level\s+([a-z ]+?)(?=\s+\w|$)", compress_fl, t, flags=re.IGNORECASE)
    def compress_rwy(m):
        ds = re.findall(rf"\b({num_words})\b", m.group(1))
        side = {"left":"L","right":"R","center":"C"}.get((m.group(2) or "").lower(), "")
        return "RWY" + "".join(WORD_TO_DIGIT[d] for d in ds) + side if ds else m.group(0)
    t = re.sub(r"\brunway\s+([a-z ]+?)\s*(left|right|center)?\b",
               compress_rwy, t, flags=re.IGNORECASE)
    t = re.sub(r"(\d+)\s+decimal\s+(\d+)", r"\1.\2", t, flags=re.IGNORECASE)
    return t.strip()

## Dataset

In [ ]:
DATASET_ID = "Jzuluaga/atco2_corpus_1h"

try:
    dataset = load_dataset(DATASET_ID, split="test", trust_remote_code=True)
except Exception:
    ds_all = load_dataset(DATASET_ID, trust_remote_code=True)
    dataset = ds_all[list(ds_all.keys())[0]]

TEXT_KEY = "text" if "text" in dataset[0] else "transcription"

## Config

In [ ]:
MODEL_ID = "jacktol/whisper-medium.en-fine-tuned-for-ATC"

RUNS = [
    {
        "run_id": "run1",
        "name": "jacktol-whisper-medium.en-ATC | beam=5",
        "model_id": MODEL_ID,
        "generate_kwargs": {
            "language": "english",
            "task": "transcribe",
            "temperature": 0.0,
            "num_beams": 5,
        },
    },
    {
        "run_id": "run2",
        "name": "jacktol-whisper-medium.en-ATC | beam=1 (greedy)",
        "model_id": MODEL_ID,
        "generate_kwargs": {
            "language": "english",
            "task": "transcribe",
            "temperature": 0.0,
            "num_beams": 1,
        },
    },
]

for run in RUNS:
    cfg = {**run, "dataset": DATASET_ID, "normalization": "normalize_for_wer"}
    path = OUTPUT_DIR / "configs" / f"config_{run['run_id']}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)

## Inference

In [ ]:
import gc

all_records = {}
_loaded_pipes = {}

def get_pipe(model_id):
    if model_id not in _loaded_pipes:
        _loaded_pipes[model_id] = hf_pipeline(
            "automatic-speech-recognition",
            model=model_id,
            device=0 if torch.cuda.is_available() else -1,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )
    return _loaded_pipes[model_id]


def run_inference(run_cfg, dataset):
    pipe = get_pipe(run_cfg["model_id"])
    records = []
    t0 = time.time()

    for i, item in enumerate(dataset):
        audio   = item["audio"]
        ref_raw = item.get(TEXT_KEY, "").strip()

        t_inf = time.time()
        out = pipe(
            {"array": audio["array"], "sampling_rate": audio["sampling_rate"]},
            generate_kwargs=run_cfg["generate_kwargs"],
        )
        inf_time = time.time() - t_inf

        ref_norm = normalize_for_wer(ref_raw)
        hyp_raw  = out["text"].strip()
        hyp_norm = normalize_for_wer(hyp_raw)

        w = wer(ref_norm, hyp_norm) if ref_norm else 0.0
        c = cer(ref_norm, hyp_norm) if ref_norm else 0.0

        records.append({
            "id"            : item.get("id", f"sample_{i:04d}"),
            "reference_raw" : ref_raw,
            "reference"     : ref_norm,
            "hypothesis_raw": hyp_raw,
            "hypothesis"    : hyp_norm,
            "wer"           : round(w, 4),
            "cer"           : round(c, 4),
            "duration_s"    : round(len(audio["array"]) / audio["sampling_rate"], 2),
            "inference_s"   : round(inf_time, 3),
        })

        if (i + 1) % 50 == 0:
            avg_wer = np.mean([r["wer"] for r in records])
            elapsed = time.time() - t0
            print(f"[{run_cfg['run_id']}] {i+1}/{len(dataset)}  WER={avg_wer:.2%}  {elapsed:.0f}s")

    return records


for run in RUNS:
    records = run_inference(run, dataset)
    all_records[run["run_id"]] = records

    out_path = OUTPUT_DIR / f"predictions_{run['run_id']}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

## Benchmark

In [ ]:
def compute_sdi(records):
    S = D = I = H = N = 0
    for r in records:
        try:
            out = process_words(r["reference"], r["hypothesis"])
            S += out.substitutions
            D += out.deletions
            I += out.insertions
            H += out.hits
            N += len(r["reference"].split())
        except Exception:
            pass
    n = max(N, 1)
    refs  = [r["reference"]  for r in records]
    hyps  = [r["hypothesis"] for r in records]
    return {
        "WER"    : round((S+D+I)/n, 4),
        "CER"    : round(cer(refs, hyps), 4),
        "S"      : S, "D": D, "I": I, "N": N,
        "S_rate" : round(S/n, 4),
        "D_rate" : round(D/n, 4),
        "I_rate" : round(I/n, 4),
    }


summary = []
print(f"{'Config':<48} {'WER':>6} {'CER':>6} {'S':>5} {'D':>5} {'I':>5}")
print("-" * 70)

for run in RUNS:
    records   = all_records[run["run_id"]]
    sdi       = compute_sdi(records)
    wer_list  = [r["wer"] for r in records]
    perfect   = sum(1 for w in wer_list if w == 0)

    row = {
        "run_id"                  : run["run_id"],
        "name"                    : run["name"],
        "model_id"                : run["model_id"],
        "beam_size"               : run["generate_kwargs"]["num_beams"],
        "samples"                 : len(records),
        **sdi,
        "wer_mean"                : round(float(np.mean(wer_list)), 4),
        "wer_median"              : round(float(np.median(wer_list)), 4),
        "wer_std"                 : round(float(np.std(wer_list)), 4),
        "perfect_transcriptions"  : perfect,
    }
    summary.append(row)
    print(f"{run['name']:<48} {sdi['WER']:>6.2%} {sdi['CER']:>6.2%}"
          f" {sdi['S']:>5} {sdi['D']:>5} {sdi['I']:>5}")

bench_path = OUTPUT_DIR / "benchmark_results.json"
with open(bench_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

## Normalization

In [ ]:
records_r1 = all_records["run1"]

refs_raw  = [r["reference_raw"].lower()  for r in records_r1]
hyps_raw  = [r["hypothesis_raw"].lower() for r in records_r1]
wer_raw   = wer(refs_raw, hyps_raw)

refs_norm = [r["reference"]  for r in records_r1]
hyps_norm = [r["hypothesis"] for r in records_r1]
wer_norm  = wer(refs_norm, hyps_norm)

improvement = wer_raw - wer_norm
print(f"WER raw : {wer_raw:.2%}")
print(f"WER norm: {wer_norm:.2%}")
print(f"delta   : {improvement:+.2%}")

examples = [
    "descend flight level one eight zero",
    "squawk four five six seven",
    "contact one two seven decimal four",
    "runway two eight left",
    "climb to flight level three five zero",
    "niner thousand feet",
    "heading two four zero",
]

norm_report = {
    "wer_without_normalization": round(wer_raw, 4),
    "wer_with_normalization"   : round(wer_norm, 4),
    "improvement"              : round(improvement, 4),
    "examples_before_after"    : [
        {"before": ex, "after_wer": normalize_for_wer(ex), "after_display": normalize_for_display(ex)}
        for ex in examples
    ],
}
norm_path = OUTPUT_DIR / "normalization_report.json"
with open(norm_path, "w", encoding="utf-8") as f:
    json.dump(norm_report, f, ensure_ascii=False, indent=2)

## Error Analysis

In [ ]:
CALLSIGN_AIRLINES = {
    "lufthansa","swiss","iberia","ryanair","easyjet","delta","united",
    "american","british","france","alitalia","klm","turkish","austrian",
    "finnair","brussels","tap","lot","aegean","croatia",
}
NUMBER_WORDS = {
    "zero","one","two","three","four","five","six","seven","eight","nine",
    "ten","eleven","twelve","thirteen","fourteen","fifteen","sixteen",
    "seventeen","eighteen","nineteen","twenty","thirty","forty","fifty",
    "sixty","seventy","eighty","ninety","hundred",
}
RUNWAY_WORDS = {"runway","approach","ils","localizer","glide"}
COMMAND_WORDS = {
    "descend","climb","maintain","heading","turn","contact","cleared",
    "squawk","frequency","report","expedite","hold","direct","ident",
}

SUGGESTIONS = {
    "callsign"     : "hot-word boosting or callsign lexicon fine-tune",
    "flight_level" : "check text normalization for alternate number forms",
    "runway"       : "runway tokens usually paired with digits — verify normalization",
    "command"      : "fixed ATC vocabulary — domain LM may help",
    "number"       : "digits are critical in ATC — consistent normalization required",
    "other"        : "check audio quality — noise or accent may be the issue",
}

def categorize(word: str) -> str:
    w = word.lower()
    if any(a in w for a in CALLSIGN_AIRLINES): return "callsign"
    if w in RUNWAY_WORDS:  return "runway"
    if w in COMMAND_WORDS: return "command"
    if w in NUMBER_WORDS:  return "number"
    if w in {"flight","level"}: return "flight_level"
    return "other"

def severity(cat: str) -> str:
    if cat in {"callsign","runway","flight_level"}: return "critical"
    if cat in {"command","number"}:                 return "moderate"
    return "minor"


errors = []
for rec in sorted(records_r1, key=lambda r: r["wer"], reverse=True):
    if len(errors) >= 35: break
    ref_w = rec["reference"].split()
    hyp_w = rec["hypothesis"].split()
    try:
        out = process_words(rec["reference"], rec["hypothesis"])
    except Exception:
        continue
    for al in out.alignments[0]:
        if len(errors) >= 35: break
        if al.type == "substitute":
            rw = ref_w[al.ref_start_idx] if al.ref_start_idx < len(ref_w) else ""
            hw = hyp_w[al.hyp_start_idx] if al.hyp_start_idx < len(hyp_w) else ""
            cat = categorize(rw)
            errors.append({
                "sample_id"   : len(errors)+1,
                "file_id"     : rec["id"],
                "error_type"  : "S",
                "category"    : cat,
                "severity"    : severity(cat),
                "ref_word"    : rw,
                "hyp_word"    : hw,
                "ref_sentence": rec["reference"],
                "hyp_sentence": rec["hypothesis"],
                "suggestion"  : SUGGESTIONS[cat],
            })
        elif al.type == "delete":
            rw = ref_w[al.ref_start_idx] if al.ref_start_idx < len(ref_w) else ""
            cat = categorize(rw)
            errors.append({
                "sample_id"   : len(errors)+1,
                "file_id"     : rec["id"],
                "error_type"  : "D",
                "category"    : cat,
                "severity"    : severity(cat),
                "ref_word"    : rw,
                "hyp_word"    : "[deleted]",
                "ref_sentence": rec["reference"],
                "hyp_sentence": rec["hypothesis"],
                "suggestion"  : SUGGESTIONS[cat],
            })

cat_counts = Counter(e["category"]  for e in errors)
sev_counts = Counter(e["severity"]  for e in errors)
typ_counts = Counter(e["error_type"] for e in errors)

print(f"samples: {len(errors)}")
print(f"types  : {dict(typ_counts)}")
print(f"cats   : {dict(cat_counts)}")
print(f"sev    : {dict(sev_counts)}")

## Save

In [ ]:
ea_json = OUTPUT_DIR / "error_analysis.json"
with open(ea_json, "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)

ea_txt = OUTPUT_DIR / "error_analysis.txt"
with open(ea_txt, "w", encoding="utf-8") as f:
    f.write("ATC-ASR Error Analysis\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"samples: {len(errors)}\n")
    f.write(f"categories: {dict(cat_counts)}\n")
    f.write(f"severity: {dict(sev_counts)}\n\n")
    f.write("=" * 60 + "\n\n")
    for e in errors:
        f.write(f"[{e['error_type']}] #{e['sample_id']} | {e['category']} | {e['severity']}\n")
        f.write(f"  REF: {e['ref_sentence']}\n")
        f.write(f"  HYP: {e['hyp_sentence']}\n")
        f.write(f"  error: '{e['ref_word']}' -> '{e['hyp_word']}'\n")
        f.write(f"  note: {e['suggestion']}\n\n")

log_path = OUTPUT_DIR / "execution_log.json"
log = {
    "timestamp"   : time.strftime("%Y-%m-%d %H:%M:%S"),
    "dataset"     : DATASET_ID,
    "samples"     : len(dataset),
    "model"       : MODEL_ID,
    "runs"        : [
        {
            "run_id"    : r["run_id"],
            "name"      : r["name"],
            "beam_size" : r["generate_kwargs"]["num_beams"],
            "wer"       : next(s["WER"] for s in summary if s["run_id"] == r["run_id"]),
            "cer"       : next(s["CER"] for s in summary if s["run_id"] == r["run_id"]),
        }
        for r in RUNS
    ],
}
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False, indent=2)